# Module 2: What Exactly is an RDD?

Objective: By the end of this chapter, students should not only know the definition of an RDD but also understand why every word in RDD exists, how Spark uses it internally, and how it differs from normal collections.


numbers = [10, 20, 30, 40, 50]

Where is this data stored?

Inside your computer’s RAM.

Everything happens inside one machine.

RDD
↓
Resilient Distributed Dataset

What Does “Resilient” Mean?

Resilient means--> Able to recover from failure.

### Why is RDD Immutable?

Because immutability:

- Makes fault recovery easier.
- Avoids conflicts when many tasks run in parallel.
- Simplifies optimization and lineage tracking.

### Advantages of RDD

- Fault tolerant
- Distributed processing
- Parallel execution
- Scalable to many machines
- Supports lazy evaluation
- Works well for low-level transformations and custom processing

### Limitations of RDD

RDDs also have drawbacks.

- No automatic query optimization.
- No schema information.
- More verbose code.
- DataFrames and Datasets are generally preferred for structured data because Spark can optimize them better.


### Full Internal Flow

Python Program
↓
SparkSession
↓
SparkContext
↓
Driver
↓
RDD Created
↓
Partitions Planned
↓
(No execution yet)
↓
collect()
↓
Job
↓
Stage
↓
Task
↓
Executor
↓
Result
↓
Driver


In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("RDD Fundamentals")
    .master("local[2]")
    .getOrCreate()
)

sc = spark.sparkContext

print(sc.appName)
print(sc.uiWebUrl)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/08 12:01:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


RDD Fundamentals
http://macbookair.lan:4040


In [ ]:
numbers=[10,20,30,40,50]

In [ ]:
rdd=sc.parallelize(numbers)
print("RDD is Created")

In [ ]:
rdd.collect()

In [ ]:
spark.stop()

# Creating RDD 

In [ ]:
spark.stop()

In [ ]:
from pyspark.sql import SparkSession
spark=(
    SparkSession.builder
    .appName("RDD Fundamentals")
    .master("local[*]")
    .getOrCreate()
)
sc=spark.sparkContext
print(sc.uiWebUrl)

sc

### Method 1

sc.parallelize() - 

In [ ]:
numbers=[1,2,3,4,5,6,7,8,9,10]

print(type(numbers))

In [ ]:
rdd = sc.parallelize(numbers,3)

In [ ]:
print(type(rdd))

In [ ]:
rdd.getNumPartitions()

In [ ]:
rdd.glom().collect()

In [ ]:
rdd.collect()

In [ ]:
rdd1=sc.parallelize(range(1,11),5)
print(rdd1.glom().collect())

### Method 2 : sc.textFile()



In [ ]:
employee_rdd=sc.textFile("employee.txt")

In [ ]:
employee_rdd.collect()

In [ ]:
employee_rdd.getNumPartitions()

In [ ]:
employee_rdd.glom().collect()

# RDD Partitions 

### What is Partition?

- A Logical chucnk of an RDD that can be processed independently 

- RDD =[1,2,3,4,5,6,7,8] - 2 PARTITIONS 

RDD 
|_ PARTITION 0 [1,2,3]
|
|- PARTITION 1 [8,8,9]

### Why Does Spark needs Partitions 

- To distribute the data across multiple nodes in a cluster for parallel processing.




In [2]:
numbers = list(range(1,21))

In [3]:
numbers

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]

In [4]:
rdd=sc.parallelize(numbers,4)

In [5]:
# Check partition count 

rdd.getNumPartitions()

4

In [ ]:
# Actual Contents inside partition 

rdd.glom().collect()

# What does Glom()() do?    
# Glom() converts each partition into a list and returns an RDD of lists.



[[1, 2, 3, 4, 5], [6, 7, 8, 9, 10], [11, 12, 13, 14, 15], [16, 17, 18, 19, 20]]

# Maximum parallel tasks at one time ---> Total avilable cores in the cluster. 

### Too Few Partition 
- Underutilization of resources.

### Too Many Partitions 
- Overhead of managing too many small tasks.

- Problems
- Task scheduling overhead 
- too much metadata 
- Excessive task launch cost 
- Possible many tiny output files 
- Driver schdeduling pressre 

### What Determines Partition Count 

- Number of cores in the cluster

- For sc.textFile() partition size depends on:

- input files 
- file sizes 
- Hadppod input slpits 
- clock/split settings 
- compression 
- filesystem 
- Spark Configuration 






In [8]:
spark.stop()

In [9]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("TooManyPartitions")
    .master("local[*]")
    .getOrCreate()
)

sc = spark.sparkContext

print("Spark Version:", sc.version)
print(sc.appName)
print(sc.uiWebUrl)
print("Available Cores:", sc.defaultParallelism)

Spark Version: 3.5.7
TooManyPartitions
http://macbookair.lan:4040
Available Cores: 8


In [13]:
rdd_normal=sc.parallelize(range(1_000_000),8)
print("Number of Partitions:", rdd_normal.getNumPartitions())

Number of Partitions: 8


In [14]:
import time 
start_time=time.time()
result=rdd_normal.map(lambda x:x*2).sum()
end_time=time.time()
print("Result:", result)
print("Time taken:", end_time - start_time)

Result: 999999000000
Time taken: 0.4452962875366211


In [16]:
rdd_many=sc.parallelize(range(1_000_000),10000)
print("Number of Partitions:", rdd_many.getNumPartitions())

Number of Partitions: 10000


In [17]:
import time 
start_time=time.time()
result=rdd_many.map(lambda x:x*2).sum()
end_time=time.time()
print("Result:", result)
print("Time taken:", end_time - start_time)

Result: 999999000000
Time taken: 152.06246519088745


In [18]:
import time
partition_count = [
    2,
    4,
    8,
    16,
    100,
    1000,
    5000
]

for partitions in partition_count:
    rdd=sc.parallelize(range(1_000_000),partitions)
    start_time=time.time()
    result=(
        rdd.map(lambda x:x*2).filter(lambda x:x%3==0).sum()
    )
    end_time=time.time()
    print(
        f"Partitions:{partitions:<5}"
        f"Time: {end_time-start_time:.4f} seconds"
    )

Partitions:2    Time: 0.4769 seconds
Partitions:4    Time: 0.2643 seconds
Partitions:8    Time: 0.4726 seconds
Partitions:16   Time: 0.4418 seconds


Partitions:100  Time: 1.8079 seconds


Partitions:1000 Time: 15.2328 seconds


Partitions:5000 Time: 77.0402 seconds


In [19]:
rdd=sc.parallelize(range(1,13),4)

In [20]:
def show_partitions(index,iterator):
    for value in iterator:
        yield(index,value)
result=rdd.mapPartitionsWithIndex(show_partitions)
result.collect()


[(0, 1),
 (0, 2),
 (0, 3),
 (1, 4),
 (1, 5),
 (1, 6),
 (2, 7),
 (2, 8),
 (2, 9),
 (3, 10),
 (3, 11),
 (3, 12)]

# RDD Lazy Evalaution and Lineage 

